# Shelf product detector (1 class) สำหรับแจ้งเตือนของหมด

โมเดล 110 คลาสเดิมเทรนจากรูปสินค้าทีละชิ้นถ่ายใกล้ ๆ จึงใช้ตัดสินว่า "ช่องบนชั้นวางว่างหรือไม่" ไม่ได้
Notebook นี้เทรน **YOLOv8n คลาสเดียว (`product`)** บน SKU-110K ซึ่งเป็นภาพชั้นวางแน่น ๆ โดยไม่แยกยี่ห้อ

| ขั้น | ส่วน | ผลลัพธ์ |
|---|---|---|
| 0 | Setup | clone repo (ใช้ `src/export_agent_assets.py`), ติดตั้งแพ็กเกจ |
| 1 | ดาวน์โหลด SKU-110K | `datasets/SKU-110K/` (ประมาณ 13.6 GB) |
| 2 | ซอยรูป | `datasets/shelf-tiles/` รูป 640×640 ที่สินค้าใหญ่ใกล้เคียงกับภาพรายช่อง |
| 3 | เทรน | `MyDrive/retail-shelf-detection/runs/detect/shelf_1class/weights/best.pt` |
| 4 | ประเมิน | mAP บน test และความแม่น "มีของ / ไม่มีของ" บนรูปของเราเอง |
| 5 | Export | `model.onnx` + `labels.json` + fixture สำหรับ retail-shelf-agent |

**วิธีรัน:** Runtime → Change runtime type → **T4 GPU** แล้ว Run all ถ้า Colab หลุดให้ Run all ใหม่ ขั้นเทรนจะทำต่อจาก `last.pt` ใน Drive

**สัญญาอนุญาต (`datasets/SKU-110K/LICENSE.txt`):** รูปของ Trax ใช้ได้เฉพาะงานวิชาการที่ไม่ใช่เชิงพาณิชย์ ห้ามแจกจ่ายรูป และห้ามดัดแปลงหรือสร้างงานดัดแปลงจากรูป
**ห้าม commit notebook ที่มี output หรือ `runs/` ขึ้น GitHub** เพราะมีภาพจาก SKU-110K ติดไปด้วย และอย่าใช้รูปจากชุดนี้เป็น fixture ใน repo สาธารณะ
ก่อนเผยแพร่โมเดลหรือใช้เชิงพาณิชย์ ต้องตรวจเงื่อนไขกับ Trax

**รูปของเราเอง (ไม่บังคับ แต่แนะนำมาก)** วางใน `MyDrive/retail-shelf-detection/` ตามนี้

```
shelf_extra/images/*.jpg  shelf_extra/labels/*.txt   เพิ่มเข้าชุด train (YOLO format, ไฟล์ label ว่าง = ไม่มีสินค้า เช่น ชั้นว่าง)
shelf_eval/filled/*.jpg   shelf_eval/empty/*.jpg     ภาพครอปรายช่องจากร้านจริง ใช้วัดผลในขั้นที่ 4 (ห้ามซ้ำกับ shelf_extra)
```

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - Runtime > Change runtime type > T4 GPU"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/ToonRz/retail-shelf-detection.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else None

if BASE is not None:
    ROOT = BASE / "retail-shelf-detection"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
%pip install -q "ultralytics>=8.3,<9" onnx onnxslim onnxruntime pyyaml matplotlib tqdm

In [ ]:
# ---------------- config ----------------
NAME, BASE_MODEL = "shelf_1class", "yolov8n.pt"
EPOCHS, IMGSZ, BATCH, PATIENCE, SEED = 30, 640, 16, 8, 0
WORKERS = 2

# Size of a tile, in "products across": tile side = median product size in that photo x this value.
# 1.5-2 looks like a crop of one shelf slot, 5 like a few facings. Picked per tile at random.
PRODUCTS_ACROSS = (1.5, 2, 3, 5)
TILES_PER_IMAGE = 4
VISIBLE_MIN = 0.3     # keep a product cut by the tile edge if at least this share of its area is visible

SAVE_TO_DRIVE = IN_COLAB and not IN_KAGGLE
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/retail-shelf-detection")
else:
    STORE = ROOT
RUNS_DIR = STORE / "runs" / "detect"
EXTRA_DIR, EVAL_DIR = STORE / "shelf_extra", STORE / "shelf_eval"
DATA_DIR = ROOT / "datasets"
TILES = DATA_DIR / "shelf-tiles"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("runs ->", RUNS_DIR)

## 1. ดาวน์โหลด SKU-110K
Ultralytics ดาวน์โหลดและแปลง label ให้เอง ถ้ามีข้อมูลแล้วจะข้าม

In [ ]:
import yaml
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.checks import check_yaml

# Absolute "path", so the data lands in DATA_DIR whatever the Ultralytics datasets_dir setting is.
cfg = yaml.safe_load(open(check_yaml("SKU-110K.yaml")))
cfg["path"] = str(DATA_DIR / "SKU-110K")
DATA_DIR.mkdir(parents=True, exist_ok=True)
sku_yaml = DATA_DIR / "SKU-110K.yaml"
sku_yaml.write_text(yaml.safe_dump(cfg))
SKU = Path(check_det_dataset(str(sku_yaml), autodownload=True)["path"])
for s in ("train", "val", "test"):
    print(s, sum(1 for _ in open(SKU / f"{s}.txt")), "images")

## 2. ซอยรูป
รูป SKU-110K มีสินค้าราว 150 ชิ้นต่อรูป ถ้าย่อทั้งรูปลง 640 สินค้าจะเล็กมาก แต่ตอนใช้งานจริงเราจะครอปภาพรายช่อง สินค้าจึงใหญ่
ขั้นนี้ครอปรอบสินค้าที่สุ่มเลือก ให้ขนาดกรอบเท่ากับสินค้า 1.5–5 ชิ้น แล้วย่อขยายเป็น 640×640

นอกจากนี้จะเก็บรูปส่วนที่ไม่มีสินค้าเลย (ถ้าหาเจอ) เป็นตัวอย่างพื้นหลัง เพื่อให้โมเดลเรียนรู้ว่าพื้นที่ว่างไม่ใช่สินค้า
SKU-110K มีภาพแบบนี้น้อย จึงแนะนำให้เติมรูปชั้นว่างจริงใน `shelf_extra`

In [ ]:
import random, shutil
from concurrent.futures import ThreadPoolExecutor
import cv2
import numpy as np
from tqdm.auto import tqdm


def read_boxes(txt, w, h):
    """YOLO label file -> pixel xyxy array (n, 4)."""
    rows = np.loadtxt(txt, ndmin=2, usecols=(1, 2, 3, 4)) if txt.exists() and txt.stat().st_size else np.zeros((0, 4))
    if not len(rows):
        return np.zeros((0, 4))
    c = rows * [w, h, w, h]
    return np.c_[c[:, 0] - c[:, 2] / 2, c[:, 1] - c[:, 3] / 2, c[:, 0] + c[:, 2] / 2, c[:, 1] + c[:, 3] / 2]


def clip_to_tile(boxes, x0, y0, S):
    """Boxes visible in the square tile (x0, y0, S), as normalized YOLO rows (cx, cy, w, h)."""
    if not len(boxes):
        return np.zeros((0, 4))
    b = boxes - [x0, y0, x0, y0]
    c = np.clip(b, 0, S)
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    vis = (c[:, 2] - c[:, 0]) * (c[:, 3] - c[:, 1])
    c = c[(area > 0) & (vis >= VISIBLE_MIN * area)]
    return np.c_[(c[:, 0] + c[:, 2]) / 2, (c[:, 1] + c[:, 3]) / 2, c[:, 2] - c[:, 0], c[:, 3] - c[:, 1]] / S


def save_tile(im, x0, y0, S, rows, img_dir, lbl_dir, stem):
    crop = im[y0:y0 + S, x0:x0 + S]
    crop = cv2.resize(crop, (IMGSZ, IMGSZ), interpolation=cv2.INTER_AREA if S > IMGSZ else cv2.INTER_LINEAR)
    cv2.imwrite(str(img_dir / f"{stem}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
    (lbl_dir / f"{stem}.txt").write_text("".join(f"0 {a:.5f} {b:.5f} {c:.5f} {d:.5f}\n" for a, b, c, d in rows))


def tile_image(path, img_dir, lbl_dir):
    """Returns (product tiles, background tiles) written for one photo."""
    im = cv2.imread(str(path))
    if im is None:                 # SKU-110K ships a few corrupt JPEGs
        return 0, 0
    h, w = im.shape[:2]
    boxes = read_boxes(path.parent.parent / "labels" / f"{path.stem}.txt", w, h)
    if not len(boxes):
        return 0, 0
    rng = random.Random(f"{SEED}:{path.name}")
    size = float(np.median(np.maximum(boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1])))
    n_pos = n_neg = 0
    for i in range(TILES_PER_IMAGE):
        S = max(64, min(int(size * rng.choice(PRODUCTS_ACROSS)), h, w))
        b = boxes[rng.randrange(len(boxes))]
        x0 = int(np.clip((b[0] + b[2]) / 2 - S * rng.uniform(0.2, 0.8), 0, w - S))
        y0 = int(np.clip((b[1] + b[3]) / 2 - S * rng.uniform(0.2, 0.8), 0, h - S))
        rows = clip_to_tile(boxes, x0, y0, S)
        if len(rows):
            save_tile(im, x0, y0, S, rows, img_dir, lbl_dir, f"{path.stem}_{i}")
            n_pos += 1
    S = max(64, min(int(size * 2), h, w))
    for _ in range(20):            # a background tile: no box touches it at all
        x0, y0 = rng.randint(0, w - S), rng.randint(0, h - S)
        if not np.any((boxes[:, 0] < x0 + S) & (boxes[:, 2] > x0) & (boxes[:, 1] < y0 + S) & (boxes[:, 3] > y0)):
            save_tile(im, x0, y0, S, [], img_dir, lbl_dir, f"{path.stem}_bg")
            n_neg += 1
            break
    return n_pos, n_neg


def tile_split(split):
    img_dir, lbl_dir = TILES / "images" / split, TILES / "labels" / split
    done = TILES / f"{split}.done"
    if done.exists():
        print(split, "already tiled:", sum(1 for _ in img_dir.iterdir()), "tiles")
        return
    shutil.rmtree(img_dir, ignore_errors=True)
    shutil.rmtree(lbl_dir, ignore_errors=True)
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)
    paths = [SKU / line.strip() for line in open(SKU / f"{split}.txt")]
    with ThreadPoolExecutor(os.cpu_count() or 2) as ex:
        res = list(tqdm(ex.map(lambda p: tile_image(p, img_dir, lbl_dir), paths), total=len(paths), desc=split))
    pos, neg = map(sum, zip(*res))
    skipped = sum(1 for p, n in res if p + n == 0)
    done.write_text("ok")
    print(f"{split}: {pos} product tiles, {neg} background tiles, {skipped} unreadable photos skipped")


for s in ("train", "val", "test"):
    tile_split(s)

In [ ]:
# Our own photos join the training set only. A photo without a label file is skipped on purpose:
# YOLO would treat it as "no products", which is wrong if someone just forgot to label it.
if EXTRA_DIR.exists():
    n = missing = 0
    for p in sorted((EXTRA_DIR / "images").iterdir()):
        lbl = EXTRA_DIR / "labels" / f"{p.stem}.txt"
        if not lbl.exists():
            missing += 1
            continue
        shutil.copy(p, TILES / "images" / "train" / f"extra_{p.name}")
        shutil.copy(lbl, TILES / "labels" / "train" / f"extra_{p.stem}.txt")
        n += 1
    print(f"added {n} photos from shelf_extra, skipped {missing} without a label file")
else:
    print("no shelf_extra folder: training on SKU-110K only")

DATA_YAML = TILES / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(TILES), "train": "images/train", "val": "images/val",
                                     "test": "images/test", "names": {0: "product"}}))
print(DATA_YAML.read_text())

In [ ]:
# Look before training: green boxes must sit on products, and products must be large in the tile.
import matplotlib.pyplot as plt

files = sorted((TILES / "images" / "train").glob("*.jpg"))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, p in zip(axes.flat, random.Random(SEED).sample(files, min(8, len(files)))):
    im = cv2.imread(str(p))
    for x1, y1, x2, y2 in read_boxes(TILES / "labels" / "train" / f"{p.stem}.txt", IMGSZ, IMGSZ).astype(int):
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(im[..., ::-1]); ax.set_title(p.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. เทรน
ใช้ transfer learning จาก COCO `yolov8n.pt` และ augmentation เหมือน notebook หลัก ถ้า Colab หลุด ให้ Run all ใหม่ จะทำต่อจาก `last.pt` และข้ามเมื่อมีไฟล์ `DONE`

In [ ]:
import torch
from ultralytics import YOLO

run = RUNS_DIR / NAME
last, best, done = run / "weights" / "last.pt", run / "weights" / "best.pt", run / "DONE"
if done.exists():
    print("already trained - skip")
else:
    finished = last.exists() and torch.load(str(last), map_location="cpu", weights_only=False).get("epoch", -1) == -1
    if finished:
        print("last.pt is a finished checkpoint")
    elif last.exists():
        YOLO(str(last)).train(resume=True)
    else:
        YOLO(BASE_MODEL).train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
            deterministic=True, project=str(RUNS_DIR), name=NAME, exist_ok=True, workers=WORKERS, plots=True,
            mosaic=1.0, close_mosaic=10, fliplr=0.5, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1,
        )
    done.write_text("ok")
print("best:", best, f"{best.stat().st_size / 1e6:.1f} MB")

## 4. ประเมิน
* **mAP บน test tiles** ใช้เช็กว่าเทรนได้จริง แต่เป็นรูปจากร้านในต่างประเทศ จึงยังไม่บอกว่าใช้กับร้านเราได้
* **มี / ไม่มีของ บนรูปของเรา** (`shelf_eval`) คือตัวเลขที่ตรงกับงานแจ้งเตือนของหมด

In [ ]:
import json

metrics = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
                              project=str(RUNS_DIR), name=f"{NAME}_test", exist_ok=True, plots=False, verbose=False)
summary = {"mAP50": float(metrics.box.map50), "mAP50_95": float(metrics.box.map),
           "precision": float(metrics.box.mp), "recall": float(metrics.box.mr)}
(run / "test_metrics.json").write_text(json.dumps(summary, indent=2))
print(summary)

In [ ]:
# Slot-level check on crops from a real store: does "any product box" agree with filled / empty?
CONF = 0.25
if (EVAL_DIR / "filled").exists() and (EVAL_DIR / "empty").exists():
    model = YOLO(str(best))
    def has_product(p):
        return len(model.predict(str(p), imgsz=IMGSZ, conf=CONF, verbose=False)[0].boxes) > 0
    filled = [has_product(p) for p in sorted((EVAL_DIR / "filled").iterdir())]
    empty = [has_product(p) for p in sorted((EVAL_DIR / "empty").iterdir())]
    slot = {"filled_total": len(filled), "filled_correct": sum(filled),
            "empty_total": len(empty), "empty_correct": len(empty) - sum(empty)}
    (run / "slot_metrics.json").write_text(json.dumps(slot, indent=2))
    print(slot)
    print("filled slot missed = false out-of-stock alert; empty slot called filled = missed alert")
else:
    print("no shelf_eval/filled + shelf_eval/empty: slot check skipped, the model is not validated on our own shelves yet")

## 5. Export สำหรับ agent
ได้ไฟล์ `shelf_1class_export.zip` ใน Drive ประกอบด้วย `model.onnx`, `labels.json`, `model.json`
ไม่สร้าง fixture จากรูป SKU-110K เพราะ agent เป็น repo สาธารณะ ให้สร้าง fixture ตอนออก release จากรูปที่แจกจ่ายได้แทน

In [ ]:
export_dir = ROOT / "export" / NAME
subprocess.run([sys.executable, "src/export_agent_assets.py", "--weights", str(best), "--dataset", "SKU-110K tiles",
                "--out", str(export_dir)], check=True)
archive = shutil.make_archive(str(STORE / f"{NAME}_export"), "zip", export_dir)
print((export_dir / "model.json").read_text())
print("saved:", archive)